# Paso 1: resolver nombre científico antes de contar identificado/no identificado

Antes de decidir qué foto "tiene especie" y cuál no, hay que intentar resolver el nombre común que ya vive en el nombre de archivo (`RAMA-AUTOR-ESPECIE-hex`) contra la lista oficial de especies de Campus Lab (`BASE_DATOS_GENERAL.xlsx`). Sin esto, cosas como "Guamuchil" o "Eucaliptolimón" cuentan como "sin identificar" cuando en realidad sí tenían nombre, solo que común, no científico.

**Categorías de salida, no binario:**
- **Resuelta**: nombre común encontrado en la lista oficial, mapea a un solo científico → se asigna directo.
- **Ambigua**: el nombre común mapea a más de un científico en la lista oficial (ej. "Algarrobo" → 3 especies distintas) → no se puede resolver sola, queda para revisión humana.
- **Código de espécimen**: el tercer segmento es un código tipo "106_020", no un nombre. Se agrupan como "mismo espécimen", útil para más adelante (confirmar una y aplicar a las demás del grupo).
- **Sin nombre reconocible**: no está en la lista oficial ni parece código. Candidata real a Pl@ntNet/iNaturalist/revisión manual.

**No mueve ni renombra archivos.** Solo decide especie_id en la base de datos, a partir de lo que el nombre de archivo ya dice.

In [1]:
import json
import re
import sqlite3
import unicodedata
from pathlib import Path
from collections import defaultdict

import pandas as pd

def cargar_config(archivo="config_local.json"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontro {archivo}. Copia config_local.example.json y ajustalo.")
    return json.loads(ruta.read_text(encoding="utf-8"))

config = cargar_config()
RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")
RUTA_EXCEL_REFERENCIA = Path(config.get("ruta_excel_base_datos_general", "BASE_DATOS_GENERAL.xlsx"))

con = sqlite3.connect(RUTA_BD)
print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")
print(f"Excel de referencia: {RUTA_EXCEL_REFERENCIA.resolve()} ({'existe' if RUTA_EXCEL_REFERENCIA.exists() else 'NO ENCONTRADO'})")

Base de datos: C:\Users\hecto\OneDrive\Escritorio\ITESO\Comunidad\Campus Lab\experimentos\bd\campuslab_volumen.db (existe)
Excel de referencia: C:\Users\hecto\OneDrive\Escritorio\ITESO\Comunidad\Campus Lab\reestructura\notebooks\BASE_DATOS_GENERAL.xlsx (existe)


## 1. Cargar y limpiar la lista oficial de especies

In [2]:
def limpiar_cientifico(nombre):
    """Quita autoria botanica/asteriscos, deja solo 'Genus species'."""
    if pd.isna(nombre):
        return None
    s = str(nombre).strip().rstrip("*")
    m = re.match(r"^([A-ZÁÉÍÓÚ][a-záéíóúñ]+(?:\s*[×x]\s*)?\s*[a-záéíóúñ.-]+)", s)
    return m.group(1).strip() if m else s

def normalizar_comun(nombre):
    """Minusculas, sin espacios ni acentos, para comparar contra el token del nombre de archivo."""
    if pd.isna(nombre):
        return None
    s = unicodedata.normalize("NFKD", str(nombre)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())

hojas = {
    "Lista_ÁRBOLES": ("Especie", "Nombre científico", "arboles", "Página naturalista", None),
    "Lista_SUCULENTAS": ("Nombre común", "Nombre científico (ID)", "suculentas", "Página naturalista", None),
    "Lista_AVES": ("Nombre común", "Nombre científico (ID)", "aves", None, "Página xeno-canto"),
}

filas_ref = []
for hoja, (col_comun, col_cientifico, grupo, col_naturalista, col_xenocanto) in hojas.items():
    df_hoja = pd.read_excel(RUTA_EXCEL_REFERENCIA, sheet_name=hoja)
    for _, row in df_hoja.iterrows():
        comun = row.get(col_comun)
        cientifico = limpiar_cientifico(row.get(col_cientifico))
        if pd.isna(comun) or not cientifico:
            continue
        filas_ref.append({
            "grupo": grupo,
            "nombre_comun": str(comun).strip(),
            "comun_normalizado": normalizar_comun(comun),
            "nombre_cientifico": cientifico,
            "pagina_naturalista": row.get(col_naturalista) if col_naturalista else (
                row.get("Página naturalista") if "Página naturalista" in df_hoja.columns else None),
            "pagina_xenocanto": row.get(col_xenocanto) if col_xenocanto else None,
        })

# Listas oficiales de ITESO que todavia no viven en BASE_DATOS_GENERAL.xlsx, sacadas directo
# de paginas/documentos de Campus Lab y validadas contra GBIF a mano. Cada CSV: 'con_comun'
# indica si trae nombre_comun real (arboles, con la columna) o solo el cientifico (orquideas,
# flores de La Primavera): en ese caso se usa el cientifico tambien como "comun" para que
# nombres de archivo con el binomio pegado (ej. "OncidiumStelligerum") resuelvan directo por
# coincidencia exacta, sin necesitar la reconstruccion heuristica de "posible_cientifico".
LISTAS_ITESO = [
    ("Lista_Orquideas_ITESO.csv", "orquideas", False),
    ("Lista_Arboles_ITESO.csv", "arboles", True),
    ("Lista_Flores_Primavera.csv", "flores_primavera", False),
]

for archivo, grupo, con_comun in LISTAS_ITESO:
    ruta = Path(archivo)
    if not ruta.exists():
        print(f"Aviso: no se encontro {archivo}, se sigue sin esa lista")
        continue
    df_lista = pd.read_csv(ruta)
    for _, row in df_lista.iterrows():
        cientifico = row["nombre_cientifico"]
        comun = row["nombre_comun"] if con_comun and pd.notna(row.get("nombre_comun")) else cientifico
        filas_ref.append({
            "grupo": grupo, "nombre_comun": comun,
            "comun_normalizado": normalizar_comun(comun), "nombre_cientifico": cientifico,
            "pagina_naturalista": row.get("fuente"), "pagina_xenocanto": None,
        })
    print(f"{archivo} cargada: {len(df_lista)} filas")

df_ref = pd.DataFrame(filas_ref).drop_duplicates(subset=["comun_normalizado", "nombre_cientifico"])
print(f"Referencias cargadas: {len(df_ref)} (de {sum(1 for h in hojas)} hojas + {len(LISTAS_ITESO)} listas ITESO)")

# Detectar ambiguedad real: mismo nombre comun normalizado -> mas de un cientifico distinto
ambiguos = df_ref.groupby("comun_normalizado")["nombre_cientifico"].nunique()
nombres_ambiguos = set(ambiguos[ambiguos > 1].index)
print(f"Nombres comunes ambiguos (mapean a 2+ especies): {len(nombres_ambiguos)}")
if nombres_ambiguos:
    ejemplos = df_ref[df_ref["comun_normalizado"].isin(list(nombres_ambiguos)[:5])][["nombre_comun", "nombre_cientifico"]]
    print(ejemplos.to_string(index=False))

Lista_Orquideas_ITESO.csv cargada: 35 filas
Lista_Arboles_ITESO.csv cargada: 290 filas
Lista_Flores_Primavera.csv cargada: 23 filas
Referencias cargadas: 559 (de 3 hojas + 3 listas ITESO)
Nombres comunes ambiguos (mapean a 2+ especies): 34
     nombre_comun  nombre_cientifico
           Ginkgo      Ginkgo biloba
 Árbol de la mora         Morus alba
 Árbol de la mora Morus celtidifolia
       Podocarpus Podocarpus reichei
     Rosal blanco           Rosa sp.
Lechuguilla Mansa     Agave filifera
Lechuguilla Mansa   Agave schidigera
           Ginkgo     Ginkgo biloba;
       Podocarpus      Podocarpus sp
     Rosal blanco            Rosa sp


## 2. Guardar la referencia en la base (tabla propia, no solo Excel)

In [3]:
con.execute("""
    CREATE TABLE IF NOT EXISTS referencia_nombres_comunes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        grupo TEXT,
        nombre_comun TEXT,
        comun_normalizado TEXT,
        nombre_cientifico TEXT,
        pagina_naturalista TEXT,
        pagina_xenocanto TEXT
    )
""")
con.execute("DELETE FROM referencia_nombres_comunes")  # se reemplaza completa cada vez que se corre, es solo espejo del Excel
con.executemany(
    "INSERT INTO referencia_nombres_comunes (grupo, nombre_comun, comun_normalizado, nombre_cientifico, pagina_naturalista, pagina_xenocanto) VALUES (?,?,?,?,?,?)",
    df_ref[["grupo", "nombre_comun", "comun_normalizado", "nombre_cientifico", "pagina_naturalista", "pagina_xenocanto"]].itertuples(index=False, name=None)
)
con.commit()
print(f"Guardado en la base: {len(df_ref)} referencias")

Guardado en la base: 559 referencias


## 3. Parsear el nombre de archivo real y clasificar

In [ ]:
PATRON_NOMBRE = re.compile(r"^(AUTORES|ESPECIE|EVENTOS|ESPACIOS)-(.+)-([0-9a-f]{6})\.[^.]+$", re.IGNORECASE)
PATRON_CODIGO_ESPECIMEN = re.compile(r"^[\d_]+$")
# Dos palabras con mayuscula pegadas (ej. "OncidiumStelligerum") suele ser un
# nombre cientifico que ya estaba en el folder original, sin espacio por el slug.
# Dos O MAS palabras con mayuscula pegadas (antes solo aceptaba exactamente 2,
# se perdian casos como "AraucariaEspinosaBrasileña" con 3 segmentos)
PATRON_CIENTIFICO_PEGADO = re.compile(r"^([A-ZÁÉÍÓÚ][a-záéíóúñ]+){2,}$")
PATRON_SEGMENTOS = re.compile(r"[A-ZÁÉÍÓÚ][a-záéíóúñ]+")
# Palabras comunes que NO son genero ni epiteto botanico, para no confundir "CasaHugo"
# con un cientifico. Se revisa contra CUALQUIER segmento (no solo el genero): se
# descubrio con "Encyclias arboretum" -> la carpeta original era Encyclia(s)
# fotografiadas EN el Arboretum, no una especie llamada "arboretum". El bug original
# solo revisaba el genero, nunca la palabra que se tomaba como epiteto.
PALABRAS_NO_TAXONOMICAS = {"casa", "nueva", "duda", "sin", "revisar", "copia", "copias", "otras",
                           "fotos", "sub", "naturalista", "concurso", "taller", "juntas",
                           "arboretum", "huerto", "jardin", "jardín", "campus",
                           "estacion", "estación", "espacio", "espacios",
                           "subir", "subira", "nat"}  # "SubiraNat" = carpeta "Subir a Naturalista", no especie

lookup_ref = {row.comun_normalizado: (row.nombre_cientifico, row.grupo) for row in df_ref.itertuples()
              if row.comun_normalizado not in nombres_ambiguos}

def parsear_nombre_archivo(nombre_archivo):
    m = PATRON_NOMBRE.match(nombre_archivo)
    if not m:
        return {"categoria": "no_parseable", "especie_token": None}
    rama, medio, _hash = m.groups()
    partes = medio.split("-", 1)
    especie_token = partes[1] if len(partes) == 2 else partes[0]

    if especie_token.upper() == "SINESP" or not especie_token:
        return {"categoria": "sin_nombre", "especie_token": especie_token}

    if PATRON_CODIGO_ESPECIMEN.fullmatch(especie_token):
        return {"categoria": "codigo_especimen", "especie_token": especie_token}

    token_norm = normalizar_comun(especie_token)  # misma funcion que el lado de la referencia, no una version aparte
    if token_norm in nombres_ambiguos:
        return {"categoria": "ambiguo", "especie_token": especie_token}
    if token_norm in lookup_ref:
        cientifico, grupo = lookup_ref[token_norm]
        return {"categoria": "resuelto", "especie_token": especie_token, "nombre_cientifico": cientifico, "grupo": grupo}

    if PATRON_CIENTIFICO_PEGADO.match(especie_token):
        segmentos = PATRON_SEGMENTOS.findall(especie_token)
        genero, resto = segmentos[0], segmentos[1:]
        if not any(s.lower() in PALABRAS_NO_TAXONOMICAS for s in segmentos):
            reconstruido = genero + " " + " ".join(s.lower() for s in resto)
            return {"categoria": "posible_cientifico", "especie_token": especie_token, "nombre_cientifico": reconstruido}

    return {"categoria": "no_reconocido", "especie_token": especie_token}


# Bug real (11-sep): este query solo filtraba especie_id IS NULL, que tambien es
# cierto para una foto RECHAZADA a mano en Revision_Especies.ipynb (el rechazo limpia
# especie_id y deja estado='pendiente_revision'). Si el nombre de archivo volvia a
# resolver al mismo nombre comun rechazado (caso real: "Cedrorojo" -> 18 fotos
# rechazadas el 8-sep como "Cedro rojo", reconfirmadas solas como "Cedrela odorata"
# al volver a correr esta celda el 11-sep, sin que nadie las revisara otra vez), la
# celda de aplicar (mas abajo) las reconfirmaba directo, pisando el rechazo humano.
# Ahora se excluye cualquier foto con un rechazo ya registrado: si alguien la rechazo,
# necesita ojo humano de nuevo, no una re-resolucion automatica silenciosa.
fotos_ya_rechazadas = {r[0] for r in con.execute(
    "SELECT DISTINCT foto_id FROM identificaciones_revisadas WHERE decision='rechazada'"
)}

df_fotos = pd.read_sql("SELECT id, ruta_actual FROM fotos WHERE especie_id IS NULL", con)
df_fotos = df_fotos[~df_fotos["id"].isin(fotos_ya_rechazadas)]
df_fotos["nombre_archivo"] = df_fotos["ruta_actual"].apply(lambda r: Path(r).name)
resultados = df_fotos["nombre_archivo"].apply(parsear_nombre_archivo).apply(pd.Series)
df_fotos = pd.concat([df_fotos, resultados], axis=1)

print(f"Excluidas por rechazo humano previo: {len(fotos_ya_rechazadas)}")
print("=== Clasificación real (reemplaza el conteo binario anterior) ===")
print(df_fotos["categoria"].value_counts())

## 4. Aplicar: asignar especie a las resueltas

In [5]:
APLICAR_CAMBIOS = True  # <-- cambialo a True cuando el conteo de arriba se vea razonable

resueltas = df_fotos[df_fotos["categoria"] == "resuelto"]
print(f"Fotos que se resolverian: {len(resueltas)}")

if not APLICAR_CAMBIOS:
    print("APLICAR_CAMBIOS esta en False. No se toco la base todavia.")
else:
    aplicadas = 0
    for _, fila in resueltas.iterrows():
        especie_row = con.execute("SELECT id FROM especies WHERE nombre_cientifico=?", (fila["nombre_cientifico"],)).fetchone()
        if especie_row:
            especie_id = especie_row[0]
        else:
            cur = con.execute(
                "INSERT INTO especies (nombre_cientifico, grupo, sugerido_por, fuente_validacion) VALUES (?,?,?,?)",
                (fila["nombre_cientifico"], fila["grupo"], "Lista_BASE_DATOS_GENERAL", "Lista oficial Campus Lab")
            )
            especie_id = cur.lastrowid
        con.execute("UPDATE fotos SET especie_id=?, estado=? WHERE id=?", (especie_id, "usable", fila["id"]))
        aplicadas += 1
    con.commit()
    print(f"Aplicadas: {aplicadas}")

Fotos que se resolverian: 55
Aplicadas: 55


## 5. Agrupación por código de espécimen (mismo individuo, varias fotos)

In [6]:
grupos = df_fotos[df_fotos["categoria"] == "codigo_especimen"].groupby("especie_token")["id"].apply(list)
print(f"Grupos de espécimen detectados: {len(grupos)}")
print(f"Fotos involucradas: {sum(len(v) for v in grupos)}")
if len(grupos):
    tam = grupos.apply(len).sort_values(ascending=False)
    print("\nGrupos más grandes:")
    print(tam.head(10))
    print("\nCuando confirmes la especie de UNA foto de un grupo en Revision_Especies.ipynb,")
    print("las demás del mismo grupo pueden heredar la misma especie automáticamente")
    print("(pendiente de conectar esa propagación al revisor, próximo paso).")

Grupos de espécimen detectados: 0
Fotos involucradas: 0


## 6. Reporte de lo que queda pendiente

In [7]:
pendientes = df_fotos[df_fotos["categoria"].isin(["ambiguo", "no_reconocido", "sin_nombre", "no_parseable", "posible_cientifico"])]
CARPETA_SALIDA = Path("../salida")
CARPETA_SALIDA.mkdir(exist_ok=True)

# Separadas: SINESP (sin ningun rastro de nombre) es un problema distinto a las que
# si traen un token pero no se pudo resolver (ambiguo/no_reconocido/no_parseable).
sin_nombre = pendientes[pendientes["categoria"] == "sin_nombre"]
con_token_sin_resolver = pendientes[pendientes["categoria"] != "sin_nombre"]

sin_nombre[["id", "nombre_archivo", "categoria"]].to_excel(CARPETA_SALIDA / "Sin_Nombre_En_Archivo.xlsx", index=False)
cols_exportar = ["id", "nombre_archivo", "categoria", "especie_token"]
if "nombre_cientifico" in con_token_sin_resolver.columns:
    cols_exportar.append("nombre_cientifico")
con_token_sin_resolver[cols_exportar].to_excel(CARPETA_SALIDA / "Nombres_No_Resueltos.xlsx", index=False)

print(f"Sin ningun nombre en el archivo (SINESP): {len(sin_nombre)} -> Sin_Nombre_En_Archivo.xlsx")
print(f"Con algun token pero no resuelto: {len(con_token_sin_resolver)} -> Nombres_No_Resueltos.xlsx")
print()
print(con_token_sin_resolver["categoria"].value_counts())

Sin ningun nombre en el archivo (SINESP): 2959 -> Sin_Nombre_En_Archivo.xlsx
Con algun token pero no resuelto: 971 -> Nombres_No_Resueltos.xlsx

categoria
no_reconocido    971
Name: count, dtype: int64
